# Get hydro data

We'll use this data in our streamflow prediction model for East Canyon, UT.

Use the 'hyriver' environment.

In [ ]:
from pynhd import NLDI
import geopandas as gpd
import pandas as pd
from supportingscripts import getData, SNOTEL_Analyzer, dataprocessing, mapping
from shapely.geometry import box, Polygon
import os
import datetime
import matplotlib.pyplot as plt
import numpy as np
import pydaymet as daymet
import warnings
warnings.filterwarnings("ignore")

## Sites for evaluation

Different sites will be used for LSTM model training, validating, and testing.

In [ ]:
# Site #1
station_id = "10133980" # NWIS id for East Canyon Creek above East Canyon Reservoir near Morgan, UT
basinname = 'EastCanyonCreek'

In [ ]:
# Site #2
station_id = '10133800'
basinname = 'EastCanyonCreek_JeremyRanch'

In [ ]:
# Site #3
station_id = '10133650'
basinname = 'EastCanyonCreek_I80'

In [ ]:
# Site #4
station_id = '10128500'
basinname = 'WeberRiver_Oakley'

In [ ]:
# Site #5
station_id = '10149400'
basinname = 'DiamondFork_Thistle'

## Download/load all data

#### SNOTEL
Load SNOTEL data that was already downloaded.

In [ ]:
#load snotel data
unprocessed_SNOTEL = {}
#read all files in the following path into the dictionary
path = 'files/SNOTEL'
for filename in os.listdir(path):
    if filename.endswith('.csv'):
        #select the name of the file between the _ and _
        name = filename.split('_')[1] 
        unprocessed_SNOTEL[name] = pd.read_csv(os.path.join(path, filename))
        #make the date a datetime object and set to the index
        unprocessed_SNOTEL[name]['Date'] = pd.to_datetime(unprocessed_SNOTEL[name]['Date'])
        unprocessed_SNOTEL[name].set_index('Date', inplace=True)
        #rename the Snow Water Equivalent (m) Start of Day Values to SWE_cm
        unprocessed_SNOTEL[name].rename(columns={'Snow Water Equivalent (m) Start of Day Values': f"{name}_SWE_cm"}, inplace=True)
        #convert SWE_m to cm
        unprocessed_SNOTEL[name][f"{name}_SWE_cm"] = unprocessed_SNOTEL[name][f"{name}_SWE_cm"] * 100
        #remove the Water_Year column
        unprocessed_SNOTEL[name].drop(columns=['Water_Year'], inplace=True)
        #we need to know how many obs for each DF, print the df name, its length, and the start/end dates
        print(f"{name}: {len(unprocessed_SNOTEL[name])} start date: {unprocessed_SNOTEL[name].index.min()} end date: {unprocessed_SNOTEL[name].index.max()}")
    


Or download new SNOTEL data

In [ ]:
#Getting basin geometry
print('Collecting basins...', end='')
basin = nldi.get_basins(station_id)
if not os.path.exists('files'):
    os.makedirs('files')
basin.to_file(f"files/{basinname}.shp")
print('done')

site_feature = nldi.getfeature_byid("nwissite", f"USGS-{station_id}")
upstream_network = nldi.navigate_byid(
    "nwissite", f"USGS-{station_id}", "upstreamMain", "flowlines", distance=9999
)

In [ ]:
# Create geodataframe of all stations
all_stations_gdf = gpd.read_file('https://raw.githubusercontent.com/egagli/snotel_ccss_stations/main/all_stations.geojson').set_index('code')
all_stations_gdf = all_stations_gdf[all_stations_gdf['csvData']==True]

# Use the polygon geometry to select snotel sites that are within the domain
gdf_in_bbox = all_stations_gdf[all_stations_gdf.geometry.within(basin.geometry.iloc[0])]

#reset index to have siteid as a column
gdf_in_bbox.reset_index(drop=False, inplace=True)

#make begin and end date a str
gdf_in_bbox['beginDate'] = [datetime.datetime.strftime(gdf_in_bbox['beginDate'][i], "%Y-%m-%d") for i in np.arange(0,len(gdf_in_bbox),1)]
gdf_in_bbox['endDate'] = [datetime.datetime.strftime(gdf_in_bbox['endDate'][i], "%Y-%m-%d") for i in np.arange(0,len(gdf_in_bbox),1)]
gdf_in_bbox

In [ ]:
# Use the getData module to retrieve data 
OutputFolder = 'files/SNOTEL'
if not os.path.exists(OutputFolder):
    os.makedirs(OutputFolder)

for i in gdf_in_bbox.index:
    getData.getSNOTELData(gdf_in_bbox.name[i], gdf_in_bbox.code[i], gdf_in_bbox.state[i], gdf_in_bbox.beginDate[i], gdf_in_bbox.endDate[i], OutputFolder)


In [ ]:
#The site with the latest start date will guide the rest
latest_start_date = max([df.index.min() for df in unprocessed_SNOTEL.values()])

#The site with the earliest end date will guide the rest
soonest_end_date = min([df.index.max() for df in unprocessed_SNOTEL.values()])
for key in unprocessed_SNOTEL.keys():
    unprocessed_SNOTEL[key] = unprocessed_SNOTEL[key][unprocessed_SNOTEL[key].index >= latest_start_date]
    unprocessed_SNOTEL[key] = unprocessed_SNOTEL[key][unprocessed_SNOTEL[key].index <= soonest_end_date]

#merge all dictionary dataframes into one larger dataframe
SNOTEL_df = pd.concat(unprocessed_SNOTEL.values(), axis=1)
#set the date index to be the index of the first dataframe in the dictionary

SNOTEL_df.head()

#### Daymet

Load met data from Daymet

In [ ]:
nldi = NLDI()

# Get geometry and ensure CRS is correct
basin = nldi.get_basins(station_id) #get basin information, we could load the files that we saved too
geometry = basin.to_crs("EPSG:4326").geometry[0] # Get the bounding box of the geometry
start_date = "1980-10-01" # Start of water year, started with a smaller range
end_date = datetime.datetime.today().strftime('%Y-%m-%d') # End date is today, but could be set to the end of the water year
var = ["prcp", "tmin", "tmax",'srad', 'swe', 'vp', 'dayl'] # Variables to fetch, precip, temperature, solar radiation, snow water equivalent, vapor pressure, day length
dates = (start_date, end_date ) # Started with a smaller range to test

In [ ]:
#Get geometry and ensure CRS is correct
basin = NLDI().get_basins(station_id)
geometry_centroid = geometry.centroid
centroid = (geometry_centroid.x, geometry_centroid.y)


#Fetch data - authentication now happens automatically via earthaccess/.netrc
# Try this simplified call first
met_df = daymet.get_bycoords(centroid, dates, variables=var)

met_df.head()

In [ ]:
# clean the dataframe, rename the columns
met_df.rename(columns={"prcp (mm/day)": "prcp_mm_day",'srad (W/m2)': "srad_W_m2", 'swe (kg/m2)': "swe_cm", "vp (Pa)": "vp_Pa", "dayl (s)": "dayl_s", "tmin (degrees C)": "tmin_C", "tmax (degrees C)": "tmax_C"}, inplace=True)
#Calculate Mean Temperature
met_df["tmean"] = (met_df.tmax_C + met_df.tmin_C) / 2
#convert swe from kg/m2 to cm, 1 kg/m2 is equivalent to 0.1 cm of water
met_df["swe_cm"] = met_df["swe_cm"] * 0.1

#set the index to name to date
met_df.index.name = "Date"

met_df.head()

#### NLDS

Load data from NLDS

In [ ]:
# Get geometry and ensure CRS is correct
basin = nldi.get_basins(station_id) #get basin information, we could load the files that we saved too
geometry = basin.to_crs("EPSG:4326").geometry[0] # Get the bounding box of the geometry
basin_polygon_coords = list(geometry.exterior.coords)

In [ ]:
#It crashes if we try to get too much data at once, so we will get daily data for a smaller range of dates
Daily_NLDAS_df1 = getData.get_NLDAS_daily(basin_polygon_coords, 
                                          begin_date='1980-01-01', 
                                          end_date='1990-01-1')
Daily_NLDAS_df1.head()

In [ ]:
#It crashes if we try to get too much data at once, so we will get daily data for a smaller range of dates
Daily_NLDAS_df2 = getData.get_NLDAS_daily(basin_polygon_coords, 
                                          begin_date='1990-01-01', 
                                          end_date='2000-01-1')
Daily_NLDAS_df2.head()

In [ ]:
#It crashes if we try to get too much data at once, so we will get daily data for a smaller range of dates
Daily_NLDAS_df3 = getData.get_NLDAS_daily(basin_polygon_coords, 
                                          begin_date='2000-01-01', 
                                          end_date='2010-01-1')
Daily_NLDAS_df3.head()

In [ ]:
#It crashes if we try to get too much data at once, so we will get daily data for a smaller range of dates
Daily_NLDAS_df4 = getData.get_NLDAS_daily(basin_polygon_coords, 
                                          begin_date='2010-01-01', 
                                          end_date='2020-01-1')
Daily_NLDAS_df4.head()

In [ ]:
#combine the four dataframes into one
Daily_NLDAS_df = pd.concat([Daily_NLDAS_df1, Daily_NLDAS_df2, Daily_NLDAS_df3, Daily_NLDAS_df4], ignore_index=False)

Daily_NLDAS_df.head()

#### USGS streamflow

Load streamflow data

In [ ]:
#get streamflow information using the NWIS id for the gage and ghte get_usgs_streamflow function in getData.py 
streamflow = getData.get_usgs_streamflow(station_id)

In [ ]:
streamflow.head()

In [ ]:
streamflow_df = dataprocessing.clean_nwis_dataframe(streamflow)
#set the index name to Date
streamflow_df.index.name = "Date"
streamflow_df.head()

#### Now put all the data together

In [ ]:
#find the latest start date and the earliest end date for SNOTEL_df, met_df, cleaned
begin_date = max([df.index.min() for df in [SNOTEL_df, met_df, streamflow_df, Daily_NLDAS_df]]) 
end_date = min([df.index.max() for df in [SNOTEL_df, met_df, streamflow_df, Daily_NLDAS_df]]) 

#clip each dataframe to have the same begin and end dates
SNOTEL_df = SNOTEL_df[(SNOTEL_df.index >= begin_date) & (SNOTEL_df.index <= end_date)]
met_df = met_df[(met_df.index >= begin_date) & (met_df.index <= end_date)]
streamflow_df = streamflow_df[(streamflow_df.index >= begin_date) & (streamflow_df.index <= end_date)]
Daily_NLDAS_df = Daily_NLDAS_df[(Daily_NLDAS_df.index >= begin_date) & (Daily_NLDAS_df.index <= end_date)]

In [ ]:
#merge the SNOTEL_df, met_df, and streamflow dataframes
Hydro_df = pd.concat([SNOTEL_df, met_df, Daily_NLDAS_df,streamflow_df], axis=1)
#put the site_no column, second to last, and streamfow column, last column, as the first two columns in the dataframe
cols = Hydro_df.columns.tolist()
cols = cols[-2:] + cols[:-2]
Hydro_df = Hydro_df[cols]
Hydro_df.head()

In [ ]:
#all of the NaN values here should be 0, fill them
Hydro_df = Hydro_df.fillna(0)
Hydro_df.head()

In [ ]:
#take a look around peak SWE to make sure we have snotel values, early season can be tricky to assess
Hydro_df.loc['2019-03-01':'2019-04-01']

#### Save the final master data file

Save our data file!

In [ ]:
#save data with station ID in the filename
#if HydroDF directory doesn't exist, create it
if not os.path.exists("files/HydroDF"):
    os.makedirs("files/HydroDF")
Hydro_df.to_csv(f"files/HydroDF/HydroDF_{basinname}_{station_id}.csv")